# BÀI TẬP PYTHON CƠ BẢN - BÀI 3

## Đề bài

Đọc file `supermarket_sales_vn.csv` và tính:

1. Tần số và tần suất khách hàng mua ở TP HCM, Hà Nội, Đà Nẵng.
2. Tần số và tần suất khách hàng Nam, Nữ.
3. Mean, median, mode của `Total` và `Rating`.


## Chuẩn bị thư viện, đường dẫn và mô hình dữ liệu


In [ ]:
import csv
import statistics
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

# File CSV được gom vào data để tách dữ liệu khỏi notebook nguồn.
DATA_PATH = Path("data") / "supermarket_sales_vn.csv"
THANH_PHO_CAN_THONG_KE = ("TP HCM", "Hà Nội", "Đà Nẵng")
GIOI_TINH_CAN_THONG_KE = ("Nam", "Nữ")


@dataclass(frozen=True)
class GiaoDich:
    """Các trường của một giao dịch cần dùng cho ba yêu cầu thống kê."""

    thanh_pho: str
    gioi_tinh: str
    tong_tien: float
    danh_gia: float

**Tóm tắt:** Mô hình `GiaoDich` chỉ giữ bốn trường thật sự cần thiết, giúp phần thống kê rõ ràng và tránh phụ thuộc vào vị trí cột trong CSV.


## Đọc và kiểm tra dữ liệu


In [ ]:
def doc_giao_dich(duong_dan=DATA_PATH):
    """Đọc CSV theo tên cột và chuyển các giá trị số sang float."""
    giao_dich = []
    with duong_dan.open("r", encoding="utf-8-sig", newline="") as tep:
        for dong in csv.DictReader(tep):
            giao_dich.append(
                GiaoDich(
                    thanh_pho=dong["City"],
                    gioi_tinh=dong["Gender"],
                    tong_tien=float(dong["Total"]),
                    danh_gia=float(dong["Rating"]),
                )
            )
    if not giao_dich:
        raise ValueError("File dữ liệu không có giao dịch nào.")
    return giao_dich


cac_giao_dich = doc_giao_dich()
print(f"Đã đọc {len(cac_giao_dich):,} giao dịch từ {DATA_PATH}.")

Đã đọc 1,000 giao dịch từ data\supermarket_sales_vn.csv.


**Tóm tắt:** `csv.DictReader` truy cập dữ liệu theo tên cột nên an toàn hơn các hằng số chỉ số; notebook cũng dừng với thông báo rõ ràng nếu file rỗng.


## Hàm thống kê dùng chung


In [ ]:
def tinh_tan_suat(tan_so, tong_so):
    """Đổi tần số sang phần trăm; trả về 0 nếu tổng mẫu bằng 0."""
    return tan_so * 100 / tong_so if tong_so else 0.0


def tim_mode(cac_gia_tri):
    """Trả về toàn bộ mode; trả danh sách rỗng nếu mọi giá trị chỉ xuất hiện một lần."""
    tan_so = Counter(cac_gia_tri)
    so_lan_lon_nhat = max(tan_so.values())
    if so_lan_lon_nhat == 1:
        return []
    return sorted(
        gia_tri
        for gia_tri, so_lan in tan_so.items()
        if so_lan == so_lan_lon_nhat
    )


def dinh_dang_mode(cac_mode):
    """Định dạng một hoặc nhiều mode để hiển thị trong báo cáo."""
    if not cac_mode:
        return "Không có mode"
    return ", ".join(f"{gia_tri:g}" for gia_tri in cac_mode)

**Tóm tắt:** Tần suất có bảo vệ trường hợp mẫu rỗng. Hàm mode trả về tất cả giá trị đồng hạng thay vì tùy ý chọn một giá trị đầu tiên.


## Tính và hiển thị kết quả


In [ ]:
def main():
    """Tính tần số, tần suất và các đại lượng trung tâm theo yêu cầu."""
    giao_dich = doc_giao_dich()
    tong_so = len(giao_dich)

    # Đếm một lần cho từng nhóm để tránh quét lại toàn bộ dữ liệu nhiều lần.
    tan_so_thanh_pho = Counter(item.thanh_pho for item in giao_dich)
    tan_so_gioi_tinh = Counter(item.gioi_tinh for item in giao_dich)

    print("THÀNH PHỐ")
    for thanh_pho in THANH_PHO_CAN_THONG_KE:
        tan_so = tan_so_thanh_pho[thanh_pho]
        print(f"- {thanh_pho}: {tan_so} lượt, {tinh_tan_suat(tan_so, tong_so):.2f}%")

    print("\nGIỚI TÍNH")
    for gioi_tinh in GIOI_TINH_CAN_THONG_KE:
        tan_so = tan_so_gioi_tinh[gioi_tinh]
        print(f"- {gioi_tinh}: {tan_so} lượt, {tinh_tan_suat(tan_so, tong_so):.2f}%")

    tong_tien = [item.tong_tien for item in giao_dich]
    danh_gia = [item.danh_gia for item in giao_dich]

    print("\nTOTAL")
    print(f"- Mean: {statistics.mean(tong_tien):.6f}")
    print(f"- Median: {statistics.median(tong_tien):.6f}")
    print(f"- Mode: {dinh_dang_mode(tim_mode(tong_tien))}")

    print("\nRATING")
    print(f"- Mean: {statistics.mean(danh_gia):.4f}")
    print(f"- Median: {statistics.median(danh_gia):.4f}")
    print(f"- Mode: {dinh_dang_mode(tim_mode(danh_gia))}")


main()

THÀNH PHỐ
- TP HCM: 490 lượt, 49.00%
- Hà Nội: 274 lượt, 27.40%
- Đà Nẵng: 236 lượt, 23.60%

GIỚI TÍNH
- Nam: 499 lượt, 49.90%
- Nữ: 501 lượt, 50.10%

TOTAL
- Mean: 322.966749
- Median: 253.848000
- Mode: 87.234, 93.744, 175.917, 189.095, 216.846, 217.633, 263.97, 276.948, 470.988, 829.08

RATING
- Mean: 6.9727
- Median: 7.0000
- Mode: 6


**Tóm tắt kết quả:**

- Thành phố: TP HCM 490 lượt (49,00%), Hà Nội 274 lượt (27,40%), Đà Nẵng 236 lượt (23,60%).
- Giới tính: Nam 499 lượt (49,90%), Nữ 501 lượt (50,10%).
- `Total`: mean 322,966749; median 253,848; có 10 mode đồng hạng.
- `Rating`: mean 6,9727; median 7,0; mode là 6,0.

## Tổng kết

Các phép tính bám đúng ba yêu cầu. Median dùng `statistics.median` nên chính xác với tập 1.000 dòng, còn mode giữ đầy đủ các giá trị đồng hạng.
